# MiniFASNetV2 Fine-tuning — Google Colab

**Standalone notebook** — không cần clone project, tất cả code đều inline.

## Cách dùng
1. Upload `dataset_minifasnet.zip` lên Google Drive
2. Chạy từng cell theo thứ tự
3. Weights tốt nhất lưu tại `best-candidate.pth`


In [ ]:
# ── CELL 1: Mount Drive & Install dependencies ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q 'lightning>=2.5,<3'
print('Done installing.')

In [ ]:
# ── CELL 2: Config — CHỈ SỬA Ở ĐÂY ─────────────────────────────────────────
import os

# Đường dẫn đến file zip trên Drive (sửa lại cho đúng)
DATASET_ZIP = '/content/drive/MyDrive/dataset_minifasnet.zip'

# Hyperparameters
SEED        = 42
EPOCHS      = 20
WARMUP      = 3      # epochs warmup classifier-only trước khi unfreeze backbone
BATCH_SIZE  = 64     # Colab GPU => batch lớn hơn
HEAD_LR     = 1e-3   # LR classifier head
FINETUNE_LR = 5e-5   # LR backbone (sau warmup)
THRESHOLD   = 0.5    # ngưỡng phân loại (calibrate sau)

# Output
WORK_DIR    = '/content/minifasnet_training'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Work dir: {WORK_DIR}')

In [ ]:
# ── CELL 3: Giải nén dataset ─────────────────────────────────────────────────
import zipfile
from pathlib import Path

DATA_DIR = Path(WORK_DIR) / 'data'

if not DATA_DIR.exists():
    print(f'Extracting {DATASET_ZIP} ...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(DATA_DIR)
    print('Done.')
else:
    print('Data already extracted.')

# Kiểm tra
for split in ['train', 'val', 'test']:
    n0 = len(list((DATA_DIR / split / '0_fake').glob('*.jpg')))
    n1 = len(list((DATA_DIR / split / '1_real').glob('*.jpg')))
    print(f'  {split:5s}: fake={n0}, real={n1}, total={n0+n1}')

In [ ]:
# ── CELL 4: Download MiniFASNetV2 pretrained weights (hash-pinned) ──────────
import hashlib
import urllib.request

COMMIT      = 'b6d5f04ad78778917853b25c778acef6d5626d15'
BASE_URL    = f'https://raw.githubusercontent.com/minivision-ai/Silent-Face-Anti-Spoofing/{COMMIT}/'
SOURCE_SHA  = 'e498c4ec5e1ddfaba62b941a126c19d65aa564999f3309661fe43ee8bf38acd7'
WEIGHTS_SHA = 'a5eb02e1843f19b5386b953cc4c9f011c3f985d0ee2bb9819eea9a142099bec0'

CACHE_DIR   = Path(WORK_DIR) / 'cache'
CACHE_DIR.mkdir(exist_ok=True)
SRC_PATH    = CACHE_DIR / 'MiniFASNet.py'
PTH_PATH    = CACHE_DIR / '2.7_80x80_MiniFASNetV2.pth'


def fetch(url, dest, expected_sha, max_bytes=5_000_000):
    if dest.exists() and hashlib.sha256(dest.read_bytes()).hexdigest() == expected_sha:
        print(f'  [cache] {dest.name}')
        return
    print(f'  Downloading {dest.name} ...')
    with urllib.request.urlopen(url, timeout=120) as r:
        data = r.read(max_bytes + 1)
    assert len(data) <= max_bytes, 'File too large'
    assert hashlib.sha256(data).hexdigest() == expected_sha, f'SHA mismatch for {dest.name}'
    dest.write_bytes(data)
    print(f'  [ok] {dest.name}')


fetch(BASE_URL + 'src/model_lib/MiniFASNet.py',
      SRC_PATH, SOURCE_SHA)
fetch(BASE_URL + 'resources/anti_spoof_models/2.7_80x80_MiniFASNetV2.pth',
      PTH_PATH, WEIGHTS_SHA, max_bytes=10_000_000)

print('Weights ready.')

In [ ]:
# ── CELL 5: Load model ───────────────────────────────────────────────────────
import importlib.util
import torch

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Reproducibility
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE} | torch={torch.__version__}')


def load_model():
    spec = importlib.util.spec_from_file_location('minifasnet_arch', SRC_PATH)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    m = mod.MiniFASNetV2(conv6_kernel=(5, 5)).cpu().eval()
    state = torch.load(PTH_PATH, map_location='cpu', weights_only=True)
    state = {k.removeprefix('module.'): v for k, v in state.items()}
    m.load_state_dict(state, strict=True)
    return m


model = load_model().to(DEVICE)
with torch.inference_mode():
    out = model(torch.zeros(2, 3, 80, 80, device=DEVICE))
    assert out.shape == (2, 3), f'Unexpected output shape: {out.shape}'

n_params = sum(p.numel() for p in model.parameters())
print(f'[OK] MiniFASNetV2 loaded. Params: {n_params:,}  Output shape: (B, 3)')

In [ ]:
# ── CELL 6: Dataset & DataLoader ────────────────────────────────────────────
import cv2
from torch.utils.data import Dataset, DataLoader

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}


class FaceDataset(Dataset):
    """Loads pre-cropped face images from 0_fake/ and 1_real/ subfolders."""

    def __init__(self, root: Path):
        self.samples = []
        for cls_dir in sorted(root.iterdir()):
            if not cls_dir.is_dir():
                continue
            label = int(cls_dir.name[0])   # '0_fake' -> 0, '1_real' -> 1
            for img_path in sorted(cls_dir.glob('*')):
                if img_path.suffix.lower() in IMG_EXTS:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(str(path))        # BGR uint8
        if img is None:
            raise ValueError(f'Cannot read: {path}')
        # Resize to 80x80, NCHW float32, values 0..255 (MiniFASNet contract)
        img = cv2.resize(img, (80, 80), interpolation=cv2.INTER_LINEAR)
        tensor = torch.from_numpy(
            np.ascontiguousarray(img.transpose(2, 0, 1), dtype=np.float32)
        )
        return tensor, torch.tensor(float(label))


train_ds = FaceDataset(DATA_DIR / 'train')
val_ds   = FaceDataset(DATA_DIR / 'val')

gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, num_workers=2, generator=gen)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, num_workers=2)

print(f'Train: {len(train_ds)} samples  ({len(train_loader)} batches)')
print(f'Val  : {len(val_ds)} samples  ({len(val_loader)} batches)')

In [ ]:
# ── CELL 7: Lightning Module + Trainer ──────────────────────────────────────
import json, csv
from datetime import datetime, timezone
from uuid import uuid4
from torch import nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, Callback
from lightning.pytorch.loggers import CSVLogger

pl.seed_everything(SEED, workers=True)


def binary_logit(logits):
    """Convert 3-class logits to binary real-vs-spoof score."""
    return logits[:, 1] - torch.logsumexp(logits[:, [0, 2]], dim=1)


criterion = nn.BCEWithLogitsLoss()


class MiniFASNetFinetuner(pl.LightningModule):
    def __init__(self, network):
        super().__init__()
        self.network = network
        self.train_total = 0.0
        self.train_count = 0
        # Freeze backbone, only train classifier head initially
        for name, p in self.network.named_parameters():
            p.requires_grad = name.startswith('prob.')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        head, backbone = [], []
        for name, p in self.network.named_parameters():
            (head if name.startswith('prob.') else backbone).append(p)
        return torch.optim.AdamW([
            {'params': head,     'lr': HEAD_LR},
            {'params': backbone, 'lr': FINETUNE_LR},
        ], weight_decay=1e-4)

    def on_train_epoch_start(self):
        self.train_total, self.train_count = 0.0, 0
        if self.current_epoch >= WARMUP:
            for p in self.network.parameters():
                p.requires_grad = True
            for g in self.trainer.optimizers[0].param_groups:
                g['lr'] = FINETUNE_LR

    def on_train_batch_start(self, batch, batch_idx):
        # Keep BN in eval mode (freeze running stats)
        for m in self.network.modules():
            if isinstance(m, nn.modules.batchnorm._BatchNorm):
                m.eval()

    def training_step(self, batch, batch_idx):
        imgs, labels = batch
        loss = criterion(binary_logit(self(imgs)), labels)
        if not torch.isfinite(loss):
            raise ValueError('Non-finite training loss')
        self.train_total += float(loss.detach()) * len(labels)
        self.train_count += len(labels)
        self.log('train_loss', loss, on_step=False, on_epoch=True, batch_size=len(labels))
        return loss

    def validation_step(self, batch, batch_idx):
        imgs, labels = batch
        loss = criterion(binary_logit(self(imgs)), labels)
        self.log('val_loss', loss, on_step=False, on_epoch=True,
                 prog_bar=True, batch_size=len(labels))


history = []


class RecordHistory(Callback):
    def on_validation_end(self, trainer, module):
        if trainer.sanity_checking:
            return
        row = dict(
            epoch=trainer.current_epoch + 1,
            train_loss=module.train_total / module.train_count,
            processed=module.train_count,
            val_loss=float(trainer.callback_metrics['val_loss']),
        )
        history.append(row)
        print(row)


print('[OK] LightningModule defined.')

In [ ]:
# ── CELL 8: Run Training ─────────────────────────────────────────────────────
run_id  = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
run_dir = Path(WORK_DIR) / f'run-{run_id}'
run_dir.mkdir(parents=True)
print(f'Run dir: {run_dir}')

# Fresh model each run
pl.seed_everything(SEED, workers=True)
model = load_model().to(DEVICE)

ckpt = ModelCheckpoint(
    dirpath=str(run_dir / 'checkpoints'),
    filename='best-{epoch:02d}',
    monitor='val_loss', mode='min',
    save_top_k=1, save_last=False,
    save_on_train_epoch_end=False,
)
logger = CSVLogger(save_dir=str(run_dir), name='lightning', version=0)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator='gpu' if DEVICE.type == 'cuda' else 'cpu',
    devices=1,
    precision='32-true',
    deterministic=True,
    gradient_clip_val=1.0,
    callbacks=[ckpt, RecordHistory()],
    logger=logger,
    default_root_dir=str(run_dir),
    num_sanity_val_steps=0,
    log_every_n_steps=1,
)

finetuner = MiniFASNetFinetuner(model)
trainer.fit(finetuner,
            train_dataloaders=train_loader,
            val_dataloaders=val_loader)

print(f'Best checkpoint: {ckpt.best_model_path}')

In [ ]:
# ── CELL 9: Lưu weights & đánh giá validation ───────────────────────────────

def evaluate(net, loader):
    net.eval()
    loss_sum, count, tp, tn, fp, fn = 0.0, 0, 0, 0, 0, 0
    with torch.inference_mode():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            scores = binary_logit(net(imgs))
            loss_sum += criterion(scores, labels).item() * len(labels)
            pred = torch.sigmoid(scores) >= THRESHOLD
            real = labels.bool()
            tp += int((pred &  real).sum()); tn += int((~pred & ~real).sum())
            fp += int((pred & ~real).sum()); fn += int((~pred &  real).sum())
            count += len(labels)
    acc = (tp + tn) / count if count else float('nan')
    apcer = fp / (fp + tn) if (fp + tn) else float('nan')
    bpcer = fn / (fn + tp) if (fn + tp) else float('nan')
    acer  = (apcer + bpcer) / 2
    return dict(loss=loss_sum/count, accuracy=acc,
                apcer=apcer, bpcer=bpcer, acer=acer,
                tp=tp, tn=tn, fp=fp, fn=fn, n=count)


# Load best checkpoint
best_state = torch.load(ckpt.best_model_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(
    {k.removeprefix('network.'): v for k, v in best_state['state_dict'].items()},
    strict=True
)
model.to(DEVICE)

val_metrics = evaluate(model, val_loader)
print('\n=== Validation Metrics ===')
for k, v in val_metrics.items():
    print(f'  {k:10s}: {v:.4f}' if isinstance(v, float) else f'  {k:10s}: {v}')

# Save best-candidate.pth
candidate_path = run_dir / 'best-candidate.pth'
torch.save(dict(
    state_dict=model.state_dict(),
    epoch=int(best_state.get('epoch', -1)) + 1,
    validation=val_metrics,
    config=dict(seed=SEED, epochs=EPOCHS, warmup=WARMUP,
                batch_size=BATCH_SIZE, head_lr=HEAD_LR,
                finetune_lr=FINETUNE_LR, threshold=THRESHOLD),
), candidate_path)
print(f'\n[OK] Saved: {candidate_path}')

In [ ]:
# ── CELL 10: Plot loss curve ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

if history:
    epochs     = [r['epoch']      for r in history]
    train_loss = [r['train_loss'] for r in history]
    val_loss   = [r['val_loss']   for r in history]

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(epochs, train_loss, label='Train loss', marker='o', markersize=4)
    ax.plot(epochs, val_loss,   label='Val loss',   marker='s', markersize=4)
    ax.axvline(WARMUP, color='gray', linestyle='--', alpha=0.6, label=f'Backbone unfreeze (epoch {WARMUP})')
    ax.set(xlabel='Epoch', ylabel='BCEWithLogits Loss',
           title='MiniFASNetV2 Fine-tuning — Loss Curve')
    ax.legend(); ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(run_dir / 'loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {run_dir}/loss.png')
else:
    print('No history to plot.')

In [ ]:
# ── CELL: Evaluate on TEST set ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Load test dataset
test_ds     = FaceDataset(DATA_DIR / 'test')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, num_workers=2)
print(f'Test : {len(test_ds)} samples  ({len(test_loader)} batches)')


def plot_confusion_matrix(metrics, title='Confusion Matrix'):
    """Rows = actual [Fake, Real], Cols = predicted [Fake, Real]."""
    mat = np.array([
        [metrics['tn'], metrics['fp']],
        [metrics['fn'], metrics['tp']],
    ], dtype=int)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(mat, cmap='Blues')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set(xticks=[0,1], yticks=[0,1],
           xticklabels=['Fake','Real'], yticklabels=['Fake','Real'],
           xlabel='Predicted', ylabel='Actual', title=title)
    cutoff = mat.max() / 2 if mat.max() > 0 else 0
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(mat[r, c]), ha='center', va='center',
                    color='white' if mat[r, c] > cutoff else 'black', fontsize=13)
    fig.tight_layout()
    save_path = run_dir / f"{title.lower().replace(' ', '_')}.png"
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return save_path


# ── Run evaluation ──────────────────────────────────────────────────────────
# Baseline (original pretrained weights — no fine-tuning)
baseline_model = load_model().to(DEVICE)
baseline_test  = evaluate(baseline_model, test_loader)

# Fine-tuned candidate
candidate_test = evaluate(model, test_loader)

# ── Print comparison table ──────────────────────────────────────────────────
print('\n' + '='*65)
print(f"{'Model':<22} {'Accuracy':>10} {'APCER':>8} {'BPCER':>8} {'ACER':>8} {'N':>6}")
print('-'*65)
for name, m in [('Baseline (pretrained)', baseline_test),
                ('Fine-tuned candidate',  candidate_test)]:
    print(f"{name:<22} {m['accuracy']:>10.4f} {m['apcer']:>8.4f} "
          f"{m['bpcer']:>8.4f} {m['acer']:>8.4f} {m['n']:>6}")
print('='*65)
print('APCER = Attack (fake) accepted as real')
print('BPCER = Real (bona fide) rejected as fake')
print('ACER  = (APCER + BPCER) / 2')

# ── Confusion matrices ──────────────────────────────────────────────────────
p_base = plot_confusion_matrix(baseline_test,  'Baseline Test Confusion Matrix')
p_cand = plot_confusion_matrix(candidate_test, 'Finetuned Test Confusion Matrix')

# ── Save test results JSON ──────────────────────────────────────────────────
test_results = dict(baseline=baseline_test, candidate=candidate_test, threshold=THRESHOLD)
(run_dir / 'final-test.json').write_text(
    json.dumps(test_results, indent=2), encoding='utf-8')
print(f'\n[OK] Test results saved: {run_dir}/final-test.json')
print(f'[OK] Confusion matrices : {p_base.name}, {p_cand.name}')


In [ ]:
# ── CELL 12: Copy results to Drive ─────────────────────────────────────────
import shutil

DRIVE_SAVE = '/content/drive/MyDrive/minifasnet_results'
import os; os.makedirs(DRIVE_SAVE, exist_ok=True)

to_copy = [
    candidate_path,
    run_dir / 'loss.png',
    run_dir / 'final-test.json',
    run_dir / 'baseline_test_confusion_matrix.png',
    run_dir / 'finetuned_test_confusion_matrix.png',
]
for src in to_copy:
    if src.exists():
        dst = shutil.copy(src, DRIVE_SAVE)
        print(f'[OK] {src.name} -> {dst}')
    else:
        print(f'[SKIP] {src.name} not found')
